In [ ]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!git clone https://github.com/QwenLM/Qwen3-VL-Embedding.git

Cloning into 'Qwen3-VL-Embedding'...
remote: Enumerating objects: 472, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 472 (delta 39), reused 24 (delta 24), pack-reused 418 (from 2)
Receiving objects: 100% (472/472), 20.62 MiB | 20.70 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [ ]:
!pip install huggingface-hub
!hf download Qwen/Qwen3-VL-Embedding-2B --local-dir ./models/Qwen3-VL-Embedding-2B
!pip install qwen-vl-utils
import sys
import os

Hint: A new version of huggingface_hub (1.28.0) is available! You are using version 1.27.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0% 0/18 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/707 [00:00<?, ?B/s]           

Fetching 18 files:   6% 1/18 [00:00<00:04,  3.87it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 707/4.26G [00:00<430:04:24, 2.75kB/s]
Reconstructing (incomplete total...):   0% 707/4.26G [00:00<430:04:26, 2.75kB/s]
Reconstructing (incomplete total...):   0% 6.23k/4.26G [00:00<430:04:25, 2.75kB/s]
Reconstructing (incomplete total...):   0% 7.80k/4.26G [00:00<430:04:25, 2.75kB/s]
Reconstructing (incomplete total

In [ ]:
%cd /content/Qwen3-VL-Embedding

/content/Qwen3-VL-Embedding


In [ ]:
!pip install munch

In [ ]:
!pip install tqdm

In [ ]:
import os
import shutil
import random
import gc

from peft import LoraConfig, get_peft_model
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from data_qwen_path import *
from itertools import cycle
from tqdm import tqdm
import numpy as np
from util_data import SUBSET_NAMES, TEMPLATES_SMALL

from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

In [ ]:
def fix_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_dataset_name_for_template(dataset):
    dataset_name = {
        "imagenet_100": "",
        "imagenet": "",
        "std10": "",
        "pets": "pet ",
        "fgvc_aircraft": "aircraft ",
        "cars": "car ",
        "eurosat": "satellite ",
        "dtd": "texture ",
        "flowers102": "flower ",
        "food101": "food ",
        "sun397": "scene ",
        "caltech101": "",
    }[dataset]
    return dataset_name

@torch.no_grad()
def get_mu_and_kappa(model, dataset):
    dataset_name = get_dataset_name_for_template(dataset)
    templates = TEMPLATES_SMALL

    all_mus =[]
    all_kappas =[]

    for class_name in SUBSET_NAMES[dataset]:
        class_texts =[]
        for template in templates:
            class_texts.append({"text": template.format(dataset_name, class_name) + "."})

        class_embs = model.process(class_texts)
        class_embs = F.normalize(class_embs, dim=-1)

        mu = class_embs.mean(dim=0)
        D = mu.shape[0]
        R = mu.norm()

        mu = mu / R
        kappa = (R * (D - R**2)) / torch.clamp(1 - R**2, min=1e-6)

        all_mus.append(mu)
        all_kappas.append(kappa)

    all_mus = torch.stack(all_mus)
    all_kappas = torch.stack(all_kappas)

    return all_mus, all_kappas

def get_image_embedding(model, images):
    image_inputs = [{"image": img} for img in images]
    image_embs = model.process(image_inputs)
    image_embs = F.normalize(image_embs, dim=-1)
    return image_embs


def get_acc(model, data_loader, mu, logit_scale, device):
    model.model.to(device)
    mu.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Evaluating"):
            labels = labels.to(device)
            image_embedding = get_image_embedding(model, images)
            # compute similarity and predict
            similarity = logit_scale * (image_embedding @ mu.T)
            preds = similarity.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def sample_tangent_gaussian(mu, kappa, num_samples, kappa_scale=1.0, kappa_max=500.0):
    C, D = mu.shape
    device = mu.device

    kappa = kappa * kappa_scale
    kappa = torch.clamp(kappa, min=1.0, max=kappa_max)

    eps = torch.randn((C, num_samples, D), device=device, dtype=mu.dtype)

    mu_expanded = mu.unsqueeze(1)
    dot_product = (eps * mu_expanded).sum(dim=-1, keepdim=True)
    eps = eps - dot_product * mu_expanded

    kappa_expanded = kappa.view(C, 1, 1)
    eps = eps / torch.sqrt(kappa_expanded)

    samples = mu_expanded + eps
    samples = F.normalize(samples, p=2, dim=-1)

    return samples

def build_text_distribution_samples(mu, kappa, num_samples=30, kappa_scale=1.0, kappa_max=5000.0):
    C, D = mu.shape
    device = mu.device
    dtype = mu.dtype
    kappa = kappa * kappa_scale
    kappa = torch.clamp(kappa, min=1.0, max=kappa_max)
    eps = torch.randn((C, num_samples, D), device=device, dtype=dtype)
    mu_expanded = mu.unsqueeze(1)
    dot_product = (eps * mu_expanded).sum(dim=-1, keepdim=True)
    eps_tangent = eps - dot_product * mu_expanded
    kappa_expanded = kappa.view(C, 1, 1)
    eps_scaled = eps_tangent / torch.sqrt(kappa_expanded + 1e-6)
    samples = mu_expanded + eps_scaled
    samples = F.normalize(samples, p=2, dim=-1)

    return samples

def logit_from_h_vectorized(logit_scale, image_feats, centroids, area_index, chosen_centroids):
    logits_base = logit_scale * (image_feats @ centroids.t())
    logits_samples = logit_scale * (image_feats @ chosen_centroids.t())
    S = chosen_centroids.shape[0]
    logits_all = logits_base.unsqueeze(0).repeat(S, 1, 1)
    logits_all[:, :, area_index] = logits_samples.t()

    return logits_all

def compute_reg_vectorized(logit_scale, area_index, samples, feats_i, centroids):
    if feats_i.shape[0] == 0:
        return torch.tensor(0.0, device=feats_i.device)

    sampled_centroids = samples[area_index]
    logits_all = logit_from_h_vectorized(logit_scale, feats_i, centroids, area_index, sampled_centroids)
    logits_flat = logits_all.reshape(-1, logits_all.size(-1))
    labels_flat = torch.full((logits_flat.size(0),), area_index, device=feats_i.device, dtype=torch.long)

    return F.cross_entropy(logits_flat, labels_flat)

def train_one_epoch_update(
    model,
    opt_h,
    scaler,
    step,
    fewshot_train_loader,
    loader_iter_G,
    lamda1,
    lamda2,
    lamda3,
    writer,
    device,
    dataset="dtd",
    logit_scale=15
):
    model.model.train()

    for real_images, real_labels in tqdm(fewshot_train_loader):
        step += 1
        synth_images, synth_labels = next(loader_iter_G)
        real_labels = real_labels.to(device)
        synth_labels = synth_labels.to(device)
        mu, kappa = get_mu_and_kappa(model, dataset)
        torch.cuda.empty_cache()
        with torch.amp.autocast('cuda'):
            real_imgs_embedding = get_image_embedding(model, real_images)
            synth_imgs_embedding = get_image_embedding(model, synth_images)
            logits_real_all = logit_scale * (real_imgs_embedding @ mu.t())
            logits_synth_all = logit_scale * (synth_imgs_embedding @ mu.t())

            samples = build_text_distribution_samples(mu)

        log_metrics = {"br": 0, "rr": 0, "bs": 0, "rs": 0}

        with torch.amp.autocast('cuda'):
            loss_real = F.cross_entropy(logits_real_all, real_labels)
            loss_synth = F.cross_entropy(logits_synth_all, synth_labels)

            total_loss = loss_real + lamda1 * loss_synth

            log_metrics["br"] = loss_real.item()
            log_metrics["bs"] = loss_synth.item()

        reg_losses =[]
        log_rr, log_rs = 0, 0

        present_classes = torch.cat([real_labels, synth_labels]).unique()
        number_of_classes = len(present_classes)

        with torch.amp.autocast('cuda'):
            for c_id_tensor in present_classes:
                c_id = c_id_tensor.item()

                r_idx = (real_labels == c_id).nonzero(as_tuple=True)[0]
                s_idx = (synth_labels == c_id).nonzero(as_tuple=True)[0]

                if len(r_idx) > 0:
                    l_reg_r = compute_reg_vectorized(
                        logit_scale=logit_scale,
                        area_index=c_id,
                        samples=samples,
                        feats_i=real_imgs_embedding[r_idx],
                        centroids=mu
                    )
                    reg_losses.append(lamda2 * l_reg_r / number_of_classes)
                    log_rr += l_reg_r.item() / number_of_classes

                if len(s_idx) > 0:
                    l_reg_s = compute_reg_vectorized(
                        logit_scale=logit_scale,
                        area_index=c_id,
                        samples=samples,
                        feats_i=synth_imgs_embedding[s_idx],
                        centroids=mu
                    )
                    reg_losses.append(lamda3 * l_reg_s / number_of_classes)
                    log_rs += l_reg_s.item() / number_of_classes

            if reg_losses:
                total_loss = total_loss + torch.sum(torch.stack(reg_losses))
        print(f'Total loss {total_loss.item()}')
        opt_h.zero_grad(set_to_none=True)
        scaler.scale(total_loss).backward()
        scaler.step(opt_h)
        scaler.update()

        writer.add_scalar("Loss_Batch/Real_Base", log_metrics["br"], step)
        writer.add_scalar("Loss_Batch/Real_Reg", log_rr, step)
        writer.add_scalar("Loss_Batch/Synth_Base", log_metrics["bs"], step)
        writer.add_scalar("Loss_Batch/Synth_Reg", log_rs, step)
        writer.add_scalar("Loss_Batch/Total", total_loss.item(), step)

        del total_loss, logits_real_all, logits_synth_all, samples

    return step

In [ ]:
#Making train/test/synth dir
os.makedirs("/content/train", exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/data_v2.zip"  -d /content/synth_data

In [ ]:
!unzip -q "/content/drive/MyDrive/checkpoints_dtd_refined_v2.zip"  -d /content/dtd_checkpoints

In [ ]:
def train_one_epoch_update(
    model,
    opt_h,
    scaler,
    step,
    fewshot_train_loader,
    loader_iter_G,
    lamda1,
    lamda2,
    lamda3,
    lamda4,
    use_l1,
    use_l2,
    use_l3,
    use_l4,
    writer,
    device,
    dataset="dtd",
    logit_scale=15,
    num_samples= 30,
    kappa_scale=1.0
):
    model.model.train()

    for real_images, real_labels in tqdm(fewshot_train_loader):

        step += 1
        real_labels = real_labels.to(device)

        # ===== REAL EMBEDDING =====
        real_imgs_embedding = get_image_embedding(model, real_images)

        # ===== SYNTH (chỉ load khi cần) =====
        if use_l1 or use_l3:
            synth_images, synth_labels = next(loader_iter_G)
            synth_labels = synth_labels.to(device)
            synth_imgs_embedding = get_image_embedding(model, synth_images)
        else:
            synth_imgs_embedding = None

        # ===== PROTOTYPE =====
        mu, kappa = get_mu_and_kappa(model, dataset)

        with torch.amp.autocast('cuda'):

            logits_real_all = logit_scale * (real_imgs_embedding @ mu.t())
            loss_real = F.cross_entropy(logits_real_all, real_labels)

            total_loss = loss_real

            # ===== SYNTH LOSS =====
            if use_l1:
                logits_synth_all = logit_scale * (synth_imgs_embedding @ mu.t())
                loss_synth = F.cross_entropy(logits_synth_all, synth_labels)
                total_loss += lamda1 * loss_synth

        # ===== SAMPLE PROTOTYPES =====
        if use_l2 or use_l3:
            samples = build_text_distribution_samples(mu, kappa, kappa_scale= kappa_scale)

        reg_losses = []
        present_classes = real_labels.unique()

        if use_l1 or use_l3:
            present_classes = torch.cat([real_labels, synth_labels]).unique()

        number_of_classes = len(present_classes)

        # ===== REGULARIZATION =====
        for c_id_tensor in present_classes:
            c_id = c_id_tensor.item()

            r_idx = (real_labels == c_id).nonzero(as_tuple=True)[0]

            # ---- REG REAL ----
            if use_l2 and len(r_idx) > 0:
                l_reg_r = compute_reg_vectorized(
                    logit_scale,
                    c_id,
                    samples,
                    real_imgs_embedding[r_idx],
                    mu
                )
                reg_losses.append(lamda2 * l_reg_r / number_of_classes)

            # ---- REG SYNTH ----
            if use_l3 and synth_imgs_embedding is not None:
                s_idx = (synth_labels == c_id).nonzero(as_tuple=True)[0]

                if len(s_idx) > 0:
                    l_reg_s = compute_reg_vectorized(
                        logit_scale,
                        c_id,
                        samples,
                        synth_imgs_embedding[s_idx],
                        mu
                    )
                    reg_losses.append(lamda3 * l_reg_s / number_of_classes)
        #separate loss
        if use_l4:
          sep_loss =  vmf_kl_approx(mu, kappa)
          total_loss += lamda4 * sep_loss
        if reg_losses:
            total_loss += torch.sum(torch.stack(reg_losses))

        # ===== OPTIM =====
        opt_h.zero_grad(set_to_none=True)
        scaler.scale(total_loss).backward()
        scaler.step(opt_h)
        scaler.update()

        writer.add_scalar("Training Loss/Real_Base", loss_real.item(), step)

        if use_l1:
            writer.add_scalar("Training Loss/Synth_Base", loss_synth.item(), step)
        if use_l2:
            writer.add_scalar("Training Loss/Real_Reg", (l_reg_r / number_of_classes).item(), step)
        if use_l3:
            writer.add_scalar("Training Loss/Synth_Reg", (l_reg_s / number_of_classes).item(), step)
        if use_l4:
            writer.add_scalar("Training Loss/Separation_Loss", sep_loss.item(), step)
        writer.add_scalar("Training Loss/Total", total_loss.item(), step)
    print("Training Loss/Total", total_loss.item())
    return step

In [ ]:
def load_best_model(model, exp_name, ckpt_path, device="cuda"):
    import os
    import torch

    #best_model_path = os.path.join(ckpt_path, f"best_model_{exp_name}.pt")
    best_model_path = ckpt_path

    #if not os.path.exists(best_model_path):
    #    raise FileNotFoundError(f"No best_model.pt found in {ckpt_path}")

    checkpoint = torch.load(best_model_path, map_location=device)

    # ⚠️ vì bạn save model.model.state_dict()
    model.model.load_state_dict(checkpoint["model_state_dict"])

    model.model.to(device)
    model.model.eval()

    print(f"Loaded best model from epoch {checkpoint['epoch']}")
    print(f"Best validation acc: {checkpoint['best_acc']*100:.2f}%")

    return model, checkpoint

#Lọc ảnh dựa trên độ tự tin

In [ ]:
n_samples_per_class = 16
n_synth_per_class = 128
n_epochs = 40
batch_size = 32
eval_batch_size = 128
logit_scale = 15
device = "cuda" if torch.cuda.is_available() else "cpu"
model_type = "qwen"
synth_train_data_dir = "/content/synth_data/synthetic_dtd_train"
dataset  = 'dtd'
fix_random_seed(0)
# ================= ABLATION SETTINGS =================
ablation_settings = [
    #{"name": "base", "use_l1": False, "use_l2": False, "use_l3": False},
    #{"name": "synthetic", "use_l1": True, "use_l2": False, "use_l3": False},
    {"name": "reg_real", "use_l1": True, "use_l2": True, "use_l3": False},
    #{"name": "reg_synth", "use_l1": True, "use_l2": False, "use_l3": True},
    #{"name": "full", "use_l1": True, "use_l2": True, "use_l3": True},
]
ablation_acc = [
     #{"name": "base", "best ACC": 0},
     #{"name": "synthetic",  "best ACC": 0},
     {"name": "reg_real",  "best ACC": 0},
     #{"name": "reg_synth",  "best ACC": 0},
     #{"name": "full",  "best ACC": 0},
]

# Shared lambda
lambda1, lambda2, lambda3, lambda4 = 0.2, 0.04, 0.02, 0.5
# number samples each area
num_samples_area = [30]
#K_acc = {5: None, 10: None, 30: None, 100: None }
K_acc = {30: None}
# upload split file into target folder if needed
fewshot_train_loader, test_loader = get_data_loader(
        real_train_data_dir='/content/',
        real_test_data_dir="/content/",
        dataset=dataset,
        bs=batch_size,
        eval_bs=eval_batch_size,
        n_img_per_cls=n_samples_per_class,
        model_type=model_type,
    )

synth_train_loader = get_synth_train_data_loader(
        synth_train_data_dir=synth_train_data_dir,
        bs=batch_size,
        n_img_per_cls=n_synth_per_class,
        dataset=dataset,
        model_type=model_type )
print(f"Number of few-shot training samples: {len(fewshot_train_loader.dataset)}")
print(f"Number of synthetic training samples: {len(synth_train_loader.dataset)}")
print(f"Number of test samples: {len(test_loader.dataset)}")

16
Number of few-shot training samples: 752
Number of synthetic training samples: 6016
Number of test samples: 2820


In [ ]:
import json
#Experiment 1: prompts chỉ dựa trên text: tên nhãn lớp; tạo prompts có tính phân biệt giữa các tên lớp được cho trước
with open("/content/dtd_lda_refined_5epoch_good.json", "r") as f:
    loaded_list = json.load(f)

In [ ]:
loaded_list[0]#

['Bands exhibit irregular, wavy outlines, diverging from strict straightness.',
 'Unequal curvature affects band continuity, making for a less rigid structure.',
 'Variations in band density and height contribute to a dynamic, non-uniform design.',
 'Varying thicknesses make some bands more prominent; others recede naturally, adding depth.',
 'Irregular edge placements disrupt some bands, producing subtle disruptions amid a generally smooth flow.',
 "The interplay of light and shadow accentuates the rounded shapes of the bands' ends.",
 'Uneven band lengths intersperse shorter and longer stretches, disrupting typical symmetry.',
 'Band placement varies intentionally, introducing asymmetry without losing overall cohesion.',
 'Bands appear ribbed or rough-edged instead of smooth, providing tactile variety over their expanse.',
 'Banded patterns have soft edges where thinner lines merge subtly, suggesting organic growth.',
 'Wavy, elongated bands interlace to form a harmonious, flowing de

In [ ]:
SUBSET_NAMES[dataset][:5]

['banded', 'blotchy', 'braided', 'bubbly', 'bumpy']

In [ ]:
@torch.no_grad()
def get_mu_and_kappa(model, dataset):
    all_mus =[]
    all_kappas =[]

    for class_idx in range(len(SUBSET_NAMES[dataset])):
        processed_text_inputs = [{"text": s} for s in loaded_list[class_idx]]
        class_embs = model.process(processed_text_inputs)
        class_embs = F.normalize(class_embs, dim=-1)#thừa

        mu = class_embs.mean(dim=0)
        D = mu.shape[0]
        R = mu.norm()

        mu = mu / R
        kappa = (R * (D - R**2)) / torch.clamp(1 - R**2, min=1e-6)

        all_mus.append(mu)
        all_kappas.append(kappa)

    all_mus = torch.stack(all_mus)
    all_kappas = torch.stack(all_kappas)

    return all_mus, all_kappas

In [ ]:
def get_acc(model, data_loader, mu, logit_scale, device):
    model.model.to(device)
    mu.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Evaluating"):
            labels = labels.to(device)
            image_embedding = get_image_embedding(model, images)
            # compute similarity and predict
            similarity = logit_scale * (image_embedding @ mu.T)
            preds = similarity.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
def get_acc(model, data_loader, mu, logit_scale, device):
    model.model.to(device)
    mu = mu.to(device)

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):

            if len(batch) == 2:
                images, labels = batch
            elif len(batch) == 3:
                images, labels, _ = batch
            else:
                raise ValueError(f"Unexpected batch size: {len(batch)}")

            labels = labels.to(device)

            image_embedding = get_image_embedding(model, images)

            similarity = logit_scale * (image_embedding @ mu.T)
            preds = similarity.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [ ]:
model = Qwen3VLEmbedder(model_name_or_path="Qwen/Qwen3-VL-Embedding-2B")
lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["qkv", "proj", "q_proj", "v_proj"], #q_proj, v_proj: Text Encoder
        lora_dropout=0.1,
        bias="none",
        task_type="FEATURE_EXTRACTION",
    )

model.model = get_peft_model(model.model, lora_config)
model.model.gradient_checkpointing_enable() # Commenting this out to fix CheckpointError

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

chat_template.jinja:   0%|          | 0.00/5.52k [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


In [ ]:
ckpt_path = '/content/dtd_checkpoints/checkpoints/dtd_cluster_refined_prompt/best_model_dtd_cluster_refined_prompt.pt'
exp_name = ''
model, ckpt = load_best_model(model, exp_name, ckpt_path, device)

mu_eval, kappa = get_mu_and_kappa(model, dataset)

mu_eval = mu_eval.to(device)
#est_acc = get_acc(model, test_loader, mu_eval, logit_scale, device)
#rint(f"Test accuracy (best real finetune model): {test_acc*100:.2f}%")
#print("="*60)

Loaded best model from epoch 25
Best validation acc: 81.62%


In [ ]:
acc = get_acc(model, synth_train_loader, mu_eval, logit_scale, device)
print(f"Synth accuracy (best real finetune model): {acc*100:.2f}%")

Evaluating: 100%|██████████| 40/40 [00:24<00:00,  1.62it/s]

Synth accuracy (best real finetune model): 38.98%


## Lọc dự trên độ tự tin

In [ ]:
from collections import defaultdict
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm


def compute_class_confidence_threshold(
    model,
    data_loader,
    mu,
    logit_scale,
    device,
):
    model.model.to(device)
    mu = mu.to(device)

    # confidence của từng class
    class_conf = defaultdict(list)

    with torch.no_grad():
        for images, labels in tqdm(
            data_loader,
            desc="Compute confidence thresholds"
        ):

            labels = labels.to(device)

            # Image embedding
            image_embedding = get_image_embedding(model, images)

            # Classification logits
            logits = logit_scale * (image_embedding @ mu.T)

            # Probability
            probs = F.softmax(logits, dim=1)

            # Confidence của nhãn dự báo
            pred_conf, pred_labels = probs.max(dim=1)

            # Lưu confidence theo class
            for y, conf in zip(labels.cpu(), pred_conf.cpu()):
                class_conf[y.item()].append(conf.item())

    # =========================================================
    # Tính Q33, Q67 theo từng class
    # =========================================================

    thresholds = {}

    for c, values in class_conf.items():
        values = np.array(values)
        thresholds[c] = {
            "q33": np.percentile(values, 33.33),
            "q67": np.percentile(values, 66.67),
            # Có thể hữu ích để phân tích
            "mean": np.mean(values),
            "median": np.median(values),
            "min": np.min(values),
            "max": np.max(values),
            "n": len(values),
        }

    return thresholds

In [ ]:
thesholds = compute_class_confidence_threshold(
    model,
    synth_train_loader,
    mu_eval,
    logit_scale,
    device
)

Compute confidence thresholds:  93%|█████████▎| 175/188 [07:47<00:34,  2.66s/it]

In [ ]:
import pandas as pd
df_thresholds = pd.DataFrame(thesholds)

print(df_thresholds)

                12          9           15          26          43  \
q33       0.264315    0.992188    0.988281    0.988281    0.950504   
q67       0.411467    0.996094    0.992188    0.992188    0.972656   
mean      0.397881    0.985809    0.957779    0.976532    0.928429   
median    0.320312    0.996094    0.992188    0.992188    0.964844   
min       0.134766    0.351562    0.168945    0.492188    0.208984   
max       0.996094    1.000000    0.996094    0.996094    0.992188   
n       128.000000  128.000000  128.000000  128.000000  128.000000   

                35          44          24          23          46  ...  \
q33       0.980469    0.945312    0.878906    0.988281    0.972656  ...   
q67       0.992188    0.980469    0.937500    0.992188    0.980469  ...   
mean      0.914543    0.902084    0.869949    0.979797    0.902756  ...   
median    0.988281    0.968750    0.906250    0.988281    0.980469  ...   
min       0.153320    0.367188    0.443359    0.519531    0.1523

In [ ]:
thesholds

{12: {'q33': np.float64(0.26431464843749997),
  'q67': np.float64(0.41146660156250003),
  'mean': np.float64(0.39788055419921875),
  'median': np.float64(0.3203125),
  'min': np.float64(0.134765625),
  'max': np.float64(0.99609375),
  'n': 128},
 9: {'q33': np.float64(0.9921875),
  'q67': np.float64(0.99609375),
  'mean': np.float64(0.985809326171875),
  'median': np.float64(0.99609375),
  'min': np.float64(0.3515625),
  'max': np.float64(1.0),
  'n': 128},
 15: {'q33': np.float64(0.98828125),
  'q67': np.float64(0.9921875),
  'mean': np.float64(0.9577789306640625),
  'median': np.float64(0.9921875),
  'min': np.float64(0.1689453125),
  'max': np.float64(0.99609375),
  'n': 128},
 26: {'q33': np.float64(0.98828125),
  'q67': np.float64(0.9921875),
  'mean': np.float64(0.976531982421875),
  'median': np.float64(0.9921875),
  'min': np.float64(0.4921875),
  'max': np.float64(0.99609375),
  'n': 128},
 43: {'q33': np.float64(0.9505042968749999),
  'q67': np.float64(0.97265625),
  'mean': 

## Chia ngưỡng tự tin thành 3 miền: H, L, M

In [ ]:
import os
import shutil
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm


def classify_generated_images_by_confidence(
    model,
    data_loader,
    mu,
    logit_scale,
    device,
    output_dir,
):
    """
    Chia ảnh sinh của mỗi class thành 3 vùng theo RANK confidence:

        Low    : bottom 1/3
        Medium : middle 1/3
        High   : top 1/3

    Ví dụ 192 ảnh/class:

        Low    = 64
        Medium = 64
        High   = 64

    Đồng thời tính Q33/Q67 của confidence để lưu lại.
    """

    model.model.to(device)
    model.model.eval()

    mu = mu.to(device)

    # =========================================================
    # 1. Tính confidence
    # =========================================================

    results = []

    with torch.no_grad():

        for images, labels, paths in tqdm(
            data_loader,
            desc="Compute confidence"
        ):

            labels = labels.to(device)

            image_embedding = get_image_embedding(
                model,
                images
            )

            logits = logit_scale * (
                image_embedding @ mu.T
            )

            probs = F.softmax(
                logits,
                dim=1
            )

            # Confidence của lớp dự báo
            pred_conf, pred_labels = probs.max(dim=1)

            for label, pred_label, conf, path in zip(
                labels.cpu(),
                pred_labels.cpu(),
                pred_conf.cpu(),
                paths
            ):

                results.append({
                    "path": str(path),
                    "label": int(label.item()),
                    "predicted_class": int(pred_label.item()),
                    "confidence": float(conf.item()),
                })

    # =========================================================
    # 2. DataFrame
    # =========================================================

    result_df = pd.DataFrame(results)

    print("\nSố lượng ảnh theo class:")
    print(
        result_df["label"]
        .value_counts()
        .sort_index()
    )

    # =========================================================
    # 3. Tạo output folders
    # =========================================================

    for region in ["Low", "Medium", "High"]:

        os.makedirs(
            os.path.join(output_dir, region),
            exist_ok=True
        )

    # =========================================================
    # 4. Chia từng class theo RANK
    # =========================================================

    thresholds = {}

    for class_id, group in result_df.groupby("label"):

        # -----------------------------------------------------
        # Sort confidence tăng dần
        # -----------------------------------------------------

        group = group.sort_values(
            "confidence",
            ascending=True
        )

        n = len(group)

        confidence_values = group[
            "confidence"
        ].to_numpy()

        # -----------------------------------------------------
        # Tính Q33 / Q67
        # -----------------------------------------------------

        q33 = float(
            pd.Series(confidence_values).quantile(
                1 / 3
            )
        )

        q67 = float(
            pd.Series(confidence_values).quantile(
                2 / 3
            )
        )

        thresholds[class_id] = {
            "q33": q33,
            "q67": q67,
            "n": n,
        }

        # -----------------------------------------------------
        # Chia theo RANK
        # -----------------------------------------------------

        low_end = n // 3
        medium_end = 2 * n // 3

        sorted_indices = group.index.tolist()

        for rank, idx in enumerate(sorted_indices):

            if rank < low_end:

                region = "Low"

            elif rank < medium_end:

                region = "Medium"

            else:

                region = "High"

            result_df.loc[
                idx,
                "region"
            ] = region

    # =========================================================
    # 5. Copy ảnh
    # =========================================================

    for _, row in tqdm(
        result_df.iterrows(),
        total=len(result_df),
        desc="Copy images"
    ):

        class_id = int(row["label"])
        region = row["region"]
        path = row["path"]

        class_dir = os.path.join(
            output_dir,
            region,
            f"{classes[class_id]}"
        )

        os.makedirs(
            class_dir,
            exist_ok=True
        )

        filename = os.path.basename(path)

        output_path = os.path.join(
            class_dir,
            filename
        )

        # -----------------------------------------------------
        # Xử lý trùng tên
        # -----------------------------------------------------

        if os.path.exists(output_path):

            name, ext = os.path.splitext(filename)

            counter = 1

            while os.path.exists(output_path):

                new_filename = (
                    f"{name}_{counter}{ext}"
                )

                output_path = os.path.join(
                    class_dir,
                    new_filename
                )

                counter += 1

        shutil.copy2(
            path,
            output_path
        )

        result_df.loc[
            result_df.index == _,
            "output_path"
        ] = output_path

    # =========================================================
    # 6. Kiểm tra
    # =========================================================

    counts = (
        result_df
        .groupby(["label", "region"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            columns=["Low", "Medium", "High"],
            fill_value=0
        )
    )

    print("\n====================================")
    print("SỐ LƯỢNG ẢNH THEO REGION")
    print("====================================")

    print(counts)

    return result_df, thresholds

In [ ]:
result_df, thresholds = classify_generated_images_by_confidence(
    model=model,
    data_loader=synth_train_loader,
    mu=mu_eval,
    logit_scale=logit_scale,
    device=device,
    output_dir="/content/synthetic_regions",
    dataset = dataset
)

Compute confidence: 100%|██████████| 188/188 [08:20<00:00,  2.66s/it]



Số lượng ảnh theo class:
label
0     128
1     128
2     128
3     128
4     128
5     128
6     128
7     128
8     128
9     128
10    128
11    128
12    128
13    128
14    128
15    128
16    128
17    128
18    128
19    128
20    128
21    128
22    128
23    128
24    128
25    128
26    128
27    128
28    128
29    128
30    128
31    128
32    128
33    128
34    128
35    128
36    128
37    128
38    128
39    128
40    128
41    128
42    128
43    128
44    128
45    128
46    128
Name: count, dtype: int64


Copy images: 100%|██████████| 6016/6016 [00:07<00:00, 824.25it/s] 



SỐ LƯỢNG ẢNH THEO REGION
region  Low  Medium  High
label                    
0        42      43    43
1        42      43    43
2        42      43    43
3        42      43    43
4        42      43    43
5        42      43    43
6        42      43    43
7        42      43    43
8        42      43    43
9        42      43    43
10       42      43    43
11       42      43    43
12       42      43    43
13       42      43    43
14       42      43    43
15       42      43    43
16       42      43    43
17       42      43    43
18       42      43    43
19       42      43    43
20       42      43    43
21       42      43    43
22       42      43    43
23       42      43    43
24       42      43    43
25       42      43    43
26       42      43    43
27       42      43    43
28       42      43    43
29       42      43    43
30       42      43    43
31       42      43    43
32       42      43    43
33       42      43    43
34       42      43    43
35       42 